In [9]:
from src.utils.preprocessor import VietnamesePreprocessor, load_teencode, load_stopwords, _clean_and_segment
import src.utils.preprocessor as preprocessor

import matplotlib.pyplot as plt 

import sys
import pandas as pd
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.notebook import tqdm

In [10]:
data_path = "../data/summary_data2603.json"

In [11]:
df = pd.read_json(data_path, lines=True)
print(df.head(3))

                                    _id  \
0  {'$oid': '69c4f58555c3997e72cd5ea2'}   
1  {'$oid': '69c4f58555c3997e72cd5ea3'}   
2  {'$oid': '69c4f58555c3997e72cd5ea4'}   

                                                 url  \
0  https://tuoitre.vn/nho-toc-bac-cho-do-ngua-mat...   
1  https://tuoitre.vn/vi-sao-giua-dam-dong-on-ao-...   
2  https://tuoitre.vn/chieu-nguy-trang-tinh-vi-cu...   

                                             content  \
0  Chuyên gia cảnh báo nhổ tóc nhiều lần có thể k...   
1  Giữa đám đông ồn ào, ta luôn nghe được giọng n...   
2  Sự thay đổi màu sắc của loài muỗm katydid tươn...   

                                             summary  \
0  Ngày 10-3, chuyên gia da liễu Desmond Tobin (I...   
1  Giáo sư Josh McDermott và cộng sự tại MIT đã k...   
2  Các nhà khoa học từ Đại học St Andrews đã phát...   

                             model_used  post_id  
0  models/gemini-3.1-flash-lite-preview        1  
1  models/gemini-3.1-flash-lite-preview        2  

In [12]:
df.shape

(3825, 6)

In [13]:
df["model_used"].unique()

<ArrowStringArray>
['models/gemini-3.1-flash-lite-preview', 'models/gemma-3-12b-it']
Length: 2, dtype: str

In [14]:
SOURCE_DF   = df          # your existing DataFrame variable
CONTENT_COL = "content"   # raw article text column
SUMMARY_COL = "summary"   # summarised text column

# Metadata columns to carry into output. [] = everything except the two text cols
KEEP_COLS = []  # e.g. ["id", "title", "label"]

TEENCODE_PATH = "../src/utils/teencode.txt"      # or None to skip
STOPWORD_PATH = None  # or None to skip
MAX_WORKERS   = 4

In [15]:
# ── 2. LOAD RESOURCES ────────────────────────────────────────────────────────
teencode_rules = load_teencode(TEENCODE_PATH) if TEENCODE_PATH else []
stopwords      = load_stopwords(STOPWORD_PATH) if STOPWORD_PATH else set()

print(f"Teencode rules : {len(teencode_rules)}")
print(f"Stopwords      : {len(stopwords)}")
print(f"Rows to process: {len(SOURCE_DF)}")

Teencode rules : 124
Stopwords      : 0
Rows to process: 3825


In [16]:
def _process_row(args: tuple) -> tuple:
    idx, content, summary = args
    proc_content = _clean_and_segment(content or "", str(idx), teencode_rules, stopwords)
    proc_summary = _clean_and_segment(summary or "", str(idx), teencode_rules, stopwords)
    return idx, proc_content, proc_summary

In [17]:
# ── 4. RUN ───────────────────────────────────────────────────────────────────
tasks = [
    (idx, row.get(CONTENT_COL, ""), row.get(SUMMARY_COL, ""))
    for idx, row in SOURCE_DF.iterrows()
]

results: dict = {}   # idx → (proc_content, proc_summary)

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(_process_row, task): task[0] for task in tasks}

    for future in tqdm(as_completed(futures), total=len(futures), desc="Preprocessing"):
        try:
            idx, proc_content, proc_summary = future.result()
            results[idx] = (proc_content, proc_summary)
        except Exception as exc:
            idx = futures[future]
            print(f"[ERROR] row {idx}: {exc}")
            results[idx] = (None, None)

print(f"\nFinished processing {len(results)} rows.")

Preprocessing:   0%|          | 0/3825 [00:00<?, ?it/s]


Finished processing 3825 rows.


In [18]:
# ── 5. BUILD OUTPUT DATAFRAME ─────────────────────────────────────────────────
raw_text_cols = {CONTENT_COL, SUMMARY_COL}
carry_cols    = KEEP_COLS if KEEP_COLS else [
    c for c in SOURCE_DF.columns if c not in raw_text_cols
]

# Brand-new DataFrame — SOURCE_DF is never mutated
df_processed = SOURCE_DF[carry_cols].copy()
df_processed["processed_content"] = [results[i][0] for i in SOURCE_DF.index]
df_processed["processed_summary"] = [results[i][1] for i in SOURCE_DF.index]

print(f"Output shape : {df_processed.shape}")
print(f"Content OK   : {df_processed['processed_content'].notna().sum()} / {len(df_processed)}")
print(f"Empty / None : {df_processed['processed_content'].isna().sum()}")
df_processed.head(3)

Output shape : (3825, 6)
Content OK   : 3825 / 3825
Empty / None : 0


,_id,url,model_used,post_id,processed_content,processed_summary
0,{'$oid': '69c4f58555c3997e72cd5ea2'},https://tuoitre.vn/nho-toc-bac-cho-do-ngua-mat...,models/gemini-3.1-flash-lite-preview,1,chuyên_gia cảnh_báo nhổ_tóc nhiều lần có_thể k...,"ngày 10-3 , chuyên_gia da_liễu desmond tobin (..."
1,{'$oid': '69c4f58555c3997e72cd5ea3'},https://tuoitre.vn/vi-sao-giua-dam-dong-on-ao-...,models/gemini-3.1-flash-lite-preview,2,"giữa đám đông ồn_ào , ta luôn nghe được giọng ...",giáo_sư josh_mcdermott và cộng_sự tại mit đã k...
2,{'$oid': '69c4f58555c3997e72cd5ea4'},https://tuoitre.vn/chieu-nguy-trang-tinh-vi-cu...,models/gemini-3.1-flash-lite-preview,3,sự thay_đổi màu_sắc của loài muỗm katydid tươn...,các nhà_khoa_học từ đại_học st_andrews đã phát...


In [19]:
sample = SOURCE_DF[[CONTENT_COL, SUMMARY_COL]].head(2).copy()
sample["processed_content"] = df_processed["processed_content"].iloc[:2].values
sample["processed_summary"] = df_processed["processed_summary"].iloc[:2].values

sample.T.style.set_properties(**{
    "white-space": "pre-wrap",
    "text-align": "left",
    "max-width": "420px",   
})

,0,1
content,"Chuyên gia cảnh báo nhổ tóc nhiều lần có thể khiến tóc không mọc lại - Ảnh: AI Việc nhổ tóc quá thường xuyên có thể khiến tóc mỏng dần và thậm chí ngừng mọc trở lại do làm tổn thương nang tóc. Đó là cảnh báo của chuyên gia da liễu Desmond Tobin (người Ireland) đăng trên báo Tempo ngày 10-3. Giáo sư Tobin cho biết trên da đầu con người có hàng triệu nang tóc, được ví như những ""nhà máy sản xuất tóc"" siêu nhỏ, và mỗi nang tóc chỉ tạo ra một sợi tóc duy nhất. Vì vậy, việc nhổ một sợi tóc không khiến nhiều sợi khác mọc lên từ cùng một nang tóc như một số người vẫn lầm tưởng. Chuyên gia này nhấn mạnh nếu nhổ tóc nhiều lần sẽ có thể khiến nang tóc tổn thương và không mọc trở lại tại vị trí đó. Tình trạng này không chỉ xảy ra với tóc trên da đầu mà còn có thể xảy ra ở các vùng khác, chẳng hạn như lông mày. Ông Tobin dẫn lại xu hướng tỉa mỏng lông mày phổ biến trong thập niên 90 của thế kỷ trước và đầu những năm 2000, khi nhiều người thường xuyên nhổ lông mày để tạo dáng mảnh. Hậu quả là không ít trường hợp nang lông bị tổn thương và lông mày không mọc lại ở những vị trí đã nhổ. Theo ông Tobin, vấn đề chính nằm ở việc nang lông bị phá hủy. Khi nhổ một sợi tóc kèm cả chân tóc, quá trình này có thể gây tổn thương nhỏ trên da đầu, thậm chí khiến toàn bộ nang tóc bị loại bỏ và không thể phục hồi. Trong khi đó, đối với tóc bạc, chuyên gia da liễu cho biết yếu tố di truyền đóng vai trò quan trọng. Tóc bạc thường vẫn mọc bình thường, thậm chí có thể mọc nhanh hoặc khỏe hơn tóc có sắc tố. Tuy nhiên, ông cũng lưu ý rằng căng thẳng kéo dài, thiếu ngủ hoặc chế độ dinh dưỡng kém có thể thúc đẩy quá trình lão hóa sinh học của cơ thể, bao gồm cả những thay đổi liên quan đến tóc.","Giữa đám đông ồn ào, ta luôn nghe được giọng người mình thương, là do tình yêu hay do điều gì khác? - Ảnh: AI Hãy tưởng tượng bạn đang ở trong một bữa tiệc náo nhiệt với tiếng nhạc, tiếng ly chén và hàng chục cuộc trò chuyện đan xen. Thế nhưng, chỉ cần người bạn thương cất lời ở phía xa, tai bạn ngay lập tức ""bắt sóng"" được. Làm thế nào bộ não có thể thực hiện một phép lọc âm siêu hạng đến thế? Câu trả lời nằm ở cơ chế ""lợi ích nhân"" của các tế bào thần kinh. Cơ chế ""khuếch đại chọn lọc"": Khi bộ não đặt ưu tiên Giáo sư Josh McDermott và các cộng sự tại MIT đã phát hiện bộ não không chỉ đơn giản là nghe thấy mọi âm thanh, mà nó chủ động điều chỉnh ""âm lượng"" cho từng nguồn phát. Khi bạn muốn nghe giọng của một người cụ thể, vỏ não thính giác sẽ thực hiện một lệnh đặc biệt: khuếch đại các đơn vị thần kinh phản ứng với đặc điểm của giọng nói đó (như cao độ, âm vực) và đồng thời giảm hoạt động của các tế bào thần kinh khác. ""Giống như bạn đang cầm một chiếc kính lúp âm thanh"", nghiên cứu sinh Ian Griffith giải thích. Nếu giọng người thương của bạn có âm vực trầm, các tế bào thần kinh nhạy với âm trầm sẽ được nhân lên với một hệ số khuếch đại lớn, trong khi các âm cao gây nhiễu xung quanh sẽ bị ""vặn nhỏ"" lại trong tiềm thức. Vị trí không gian: ""La bàn"" thính giác Nghiên cứu của MIT cũng chỉ ra rằng bộ não là một chuyên gia về định vị không gian. Khả năng tập trung của bạn sẽ đạt hiệu quả cao nhất khi giọng nói mục tiêu và các nguồn âm thanh gây nhiễu nằm ở các vị trí khác nhau trên mặt phẳng ngang (trái - phải). Tuy nhiên, một phát hiện thú vị từ mô hình tính toán của MIT là con người gặp khó khăn hơn nhiều trong việc phân tách âm thanh nếu chúng bị chồng lấn theo mặt phẳng thẳng đứng (trên - dưới). Điều này giải thích tại sao bạn dễ dàng nói chuyện với một người đứng cạnh mình hơn là cố gắng nghe một tiếng gọi từ tầng trên giữa một căn phòng náo động. Từ phòng thí nghiệm đến hy vọng cho người khiếm thính Không chỉ dừng lại ở việc giải mã một hiện tượng tâm lý thú vị, nghiên cứu từ MIT còn mang ý nghĩa nhân văn sâu sắc, hứa hẹn tạo nên một cuộc cách mạng trong lĩnh vực hỗ trợ người khiếm thính. Đối với những người sử dụng thiết bị cấy ghép ốc tai điện tử hiện nay, môi trường ồn ào như nhà hàng hay phố xá đông đúc vẫn là một ""cơn ác mộng"". Các th

In [20]:
df_processed.to_json("../data/processed/df_processed2603.jsonl", orient="records", lines=True, force_ascii=False)